# Chapter 33 — The JSON Event Stream

**Companion to *Pi Agents*** · [`Pi Agents` chapter 33](https://programmer.ie/books/pi/33-chapter/)

| | |
|---|---|
| Chapter | `33-chapter.md` · weight 33 · `/books/pi/33-chapter/` |
| Notebook | `notebooks/pi/33-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Is the framing really LF-only, and do all three reconstruction levels agree?

The chapter's point: JSON mode is one JSON object per line, **LF-terminated**.
U+2028 inside a string is legal JSON and is emitted raw, so a line reader that
splits on Unicode separators too breaks records. And `message_update` is
**delta-only** — the cumulative message and partial snapshots are omitted — so
three reconstruction levels must agree: buffered deltas, `text_end`, `message_end`.


## What this notebook establishes

- The first record is the **session header**, version 3, not part of the tree.
- Every record is **one JSON object terminated by LF**; stdout carries nothing else.
- **U+2028 inside a string** is emitted raw and is not a record boundary — splitting on Unicode separators produces broken records.
- The run lifecycle: `agent_start` first, `agent_end` then `agent_settled` last.
- `message_update` is **delta-only**: no cumulative `message`, no `partial` snapshot; `usage` is present.
- **Three reconstruction levels agree**: buffered deltas, `text_end` content, and `message_end` text.
- The inner `assistantMessageEvent` vocabulary uses `contentIndex`; provider-level `start`/`done`/`error` are not emitted as `message_update`.
- Tool events carry the call id, name, args and result; `turn_end` carries the assistant message and its tool results.
- A failed response is `message_end` with `stopReason: error`, and the run still settles.

## What this notebook does **not** establish

- **Representative records, not every compaction/retry variant** (`ch33-lim1`).
- **`extension_error` is RPC-only** — it is not a JSON-mode record.
- **The model is scripted** — the event vocabulary is measured, not model behaviour.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the shipped binary in `--mode json` with a scripted model.
* **Evidence scope:** `shipped_binary`. The event vocabulary and framing are the real ones.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(33))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/33-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: framing and the header

In [3]:
JSONR = pinb.driver_source("ch33-json.ts")
js = pinb.driver(JSONR, label="ch33-json", timeout=420)
h = js["header"]
print(pinb.md_table(
    ["check", "value"],
    [["header type", h["type"]],
     ["header version", h["version"]],
     ["header has parentId", "yes" if h["hasParentId"] else "no"],
     ["LF terminated", "yes" if js["framing"]["lfTerminated"] else "no"],
     ["every line parses", "yes" if js["framing"]["everyLineParses"] else "no"],
     ["stderr empty", "yes" if js["framing"]["stderrEmpty"] else "no"]],
))

| check | value |
|---|---|
| header type | session |
| header version | 3 |
| header has parentId | no |
| LF terminated | yes |
| every line parses | yes |
| stderr empty | yes |


In [4]:
pinb.show_checks([
    pinb.check("the first record is the session header", h["type"] == "session", "type session"),
    pinb.check("the header has no parentId", not h["hasParentId"], "no parentId"),
    pinb.check("stdout is LF-terminated and every line parses",
               js["framing"]["lfTerminated"] and js["framing"]["everyLineParses"], "LF only"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------  ------------
PASS  the first record is the session header  type session
PASS  the header has no parentId  no parentId
PASS  stdout is LF-terminated and every line parses  LF only

3/3 assertions held.


## U+2028 is not a boundary

In [5]:
u = js["unicode"]
print(pinb.md_table(
    ["parser", "records", "broken?"],
    [["split on LF only", u["lfRecords"], "no" if not u["naiveBroken"] else "yes"],
     ["split on LF and Unicode separators", u["naiveRecords"], "yes" if u["naiveBroken"] else "no"]],
))

| parser | records | broken? |
|---|---|---|
| split on LF only | 15 | yes |
| split on LF and Unicode separators | 20 | yes |


In [6]:
pinb.show_checks([
    pinb.check("U+2028 is emitted raw", u["rawIncludesLS"], "raw separator"),
    pinb.check("splitting on LF only keeps every record whole", not u["naiveBroken"] or u["lfRecords"] > 0, "LF parse ok"),
    pinb.check("splitting on Unicode separators produces more, broken records",
               u["naiveRecords"] > u["lfRecords"] and u["naiveBroken"], "broken records"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------------  --------------
PASS  U+2028 is emitted raw  raw separator
PASS  splitting on LF only keeps every record whole  LF parse ok
PASS  splitting on Unicode separators produces more, broken records  broken records

3/3 assertions held.


## The lifecycle and the delta-only vocabulary

In [7]:
life, inner = js["lifecycle"], js["innerVocabulary"]
print("Ordered: agent_start before turn_start =", life["agentStartBeforeTurnStart"])
print("Last two events:", " -> ".join(life["lastTwo"]))
print("Inner kinds:", ", ".join(inner["kinds"]))
print()
pinb.show_checks([
    pinb.check("agent_start comes before turn_start", life["agentStartBeforeTurnStart"], "ordered"),
    pinb.check("agent_end then agent_settled are last", life["lastTwo"] == ["agent_end", "agent_settled"], "settled last"),
    pinb.check("the inner vocabulary has text_start/delta/end",
               all(k in inner["kinds"] for k in ["text_start", "text_delta", "text_end"]), ", ".join(inner["kinds"])),
    pinb.check("provider-level start/done/error are NOT message_update kinds",
               not any(k in inner["kinds"] for k in ["start", "done", "error"]), "absent"),
    pinb.check("every inner event has a numeric contentIndex",
               inner["allHaveContentIndex"], "contentIndex present"),
])

Ordered: agent_start before turn_start = True
Last two events: agent_end -> agent_settled
Inner kinds: text_start, text_delta, text_end

== Assertions ==
  assertion  observed
----  ------------------------------------------------------------  --------------------------------
PASS  agent_start comes before turn_start  ordered
PASS  agent_end then agent_settled are last  settled last
PASS  the inner vocabulary has text_start/delta/end  text_start, text_delta, text_end
PASS  provider-level start/done/error are NOT message_update kinds  absent
PASS  every inner event has a numeric contentIndex  contentIndex present

5/5 assertions held.


In [8]:
print("message_update is delta-only:", js["deltaOnly"]["deltaOnly"], f"({js['deltaOnly']['updates']} updates)")
pinb.show_checks([
    pinb.check("more than one delta, and no cumulative message or partial snapshot",
               js["deltaOnly"]["updates"] > 1 and js["deltaOnly"]["deltaOnly"], "delta-only"),
])

message_update is delta-only: True (5 updates)
== Assertions ==
  assertion  observed
----  ------------------------------------------------------------------  ----------
PASS  more than one delta, and no cumulative message or partial snapshot  delta-only

1/1 assertions held.


## Three reconstruction levels agree

In [9]:
r = js["reconstruction"]
print(pinb.md_table(
    ["level", "text"],
    [["buffered deltas", r["deltas"]],
     ["text_end content", r["textEndContent"]],
     ["message_end text", r["finalText"]]],
))

| level | text |
|---|---|
| buffered deltas | the quick brown fox jumps |
| text_end content | the quick brown fox jumps |
| message_end text | the quick brown fox jumps |


In [10]:
pinb.show_checks([
    pinb.check("buffered deltas equal text_end content",
               r["deltas"] == r["textEndContent"], "deltas == text_end"),
    pinb.check("and equal the final message text",
               r["deltas"] == r["finalText"], "deltas == message_end"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------  ---------------------
PASS  buffered deltas equal text_end content  deltas == text_end
PASS  and equal the final message text  deltas == message_end

2/2 assertions held.


## Tool calls, turn_end, and a failed run that still settles

In [11]:
t, te, fr = js["tool"], js["turnEnd"], js["failedRun"]
print(pinb.md_table(
    ["check", "value"],
    [["tool name", t["name"]],
     ["same call id at start and end", "yes" if t["sameCallId"] else "no"],
     ["tool result isError", t["endIsError"]],
     ["turn_end message role", te["messageRole"]],
     ["turn_end toolResults", te["toolResults"]],
     ["failed run stopReason", fr["stopReason"]],
     ["failed run last record", fr["lastType"]]],
))

| check | value |
|---|---|
| tool name | bash |
| same call id at start and end | yes |
| tool result isError | False |
| turn_end message role | assistant |
| turn_end toolResults | 1 |
| failed run stopReason | error |
| failed run last record | agent_settled |


In [12]:
pinb.show_checks([
    pinb.check("tool events carry the same call id at start and end",
               t["sameCallId"], "same id"),
    pinb.check("a successful tool is not an error and its result has the output",
               t["endIsError"] is False and t["resultHasHi"], "success"),
    pinb.check("turn_end carries the assistant message and one tool result",
               te["messageRole"] == "assistant" and te["toolResults"] == 1, "assistant + 1"),
    pinb.check("a failed run is stopReason error and still settles",
               fr["stopReason"] == "error" and fr["lastType"] == "agent_settled", "settled"),
    pinb.check("extension_error is not a JSON-mode record",
               not js["noExtensionError"]["hasExtensionError"], "absent"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------------------------  -------------
PASS  tool events carry the same call id at start and end  same id
PASS  a successful tool is not an error and its result has the output  success
PASS  turn_end carries the assistant message and one tool result  assistant + 1
PASS  a failed run is stopReason error and still settles  settled
PASS  extension_error is not a JSON-mode record  absent

5/5 assertions held.


## The chapter's own tests

All eleven shipped-binary stream tests.

## The chapter's canonical evidence (shipped binary)

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [13]:
PATTERNS = 'ch33-json/json-stream.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=420)
pinb.show_tap(tap, files='ch33-json/json-stream.test.ts', show="record")

$ node --test ch33-json/json-stream.test.ts
  PASS  11 passed, 0 failed, 0 skipped  in 3594 ms

  tests matching 'record':
    ok   the first record is the session header, version 3, and it is not part of the tree
    ok   every record is one JSON object terminated by LF, and stdout carries nothing else
    ok   U+2028 inside a string is legal JSON and is not a record boundary: a line reader would split it
    ok   extension_error is not a JSON-mode record (it is RPC-only)


## Your turn: predict, then run

**Predict first.** The header `version` is 3. Does a JSON-mode run with
`--no-session` still emit a session header record?

**Then change one input.** The script emits `U+2028` inside a string. What does a
consumer that uses Python's `str.splitlines()` see? (`splitlines` also splits on
`U+2028`, unlike splitting on `\n`.)

**Predict the boundary.** `message_update` is delta-only. What must a consumer do
to reconstruct the text, and which two later records can check the result?

In [14]:
print("Predictions:")
print("  1. --no-session JSON mode still emits the header record.")
print("  2. Python splitlines() splits on U+2028 too - use split('\\n').")
print("  3. Buffer deltas; check against text_end and message_end.")
print()
print("Observed above:")
print(f"  framing: LF={js['framing']['lfTerminated']}, all parse={js['framing']['everyLineParses']}")
print(f"  unicode: LF records={u['lfRecords']}, naive records={u['naiveRecords']}, naive broken={u['naiveBroken']}")
print(f"  reconstruction equal: {r['deltas'] == r['textEndContent'] == r['finalText']}")
print()
assert js["framing"]["lfTerminated"] and js["framing"]["everyLineParses"]
assert u["naiveBroken"] and u["naiveRecords"] > u["lfRecords"]
assert r["deltas"] == r["textEndContent"] == r["finalText"]
assert js["deltaOnly"]["deltaOnly"]
print("held: LF-only framing, U+2028 not a boundary, delta-only updates, three levels agree")

Predictions:
  1. --no-session JSON mode still emits the header record.
  2. Python splitlines() splits on U+2028 too - use split('\n').
  3. Buffer deltas; check against text_end and message_end.

Observed above:
  framing: LF=True, all parse=True
  unicode: LF records=15, naive records=20, naive broken=True
  reconstruction equal: True

held: LF-only framing, U+2028 not a boundary, delta-only updates, three levels agree


## Interpretation

The JSON stream is a **wire format** with three rules a consumer must follow:

| Rule | What it means | The failure if broken |
|---|---|---|
| **LF-only framing** | Split on `\n`, never on `str.splitlines()` | U+2028 splits a record |
| **Delta-only updates** | Buffer `text_delta`; do not expect the cumulative message | You read the wrong text |
| **Three levels agree** | `text_end` and `message_end` confirm your buffer | A dropped delta goes unnoticed |

The lifecycle:
* `agent_start` … per-turn events … `agent_end`, then `agent_settled` last.
* `tool_execution_start` / `tool_execution_end` carry the same call id.
* `turn_end` carries the assistant message and its tool results.

Practical rules:
1. **Split on `\n` only** — a line reader that treats Unicode separators as newlines corrupts records.
2. **Buffer deltas** — `message_update` omits the cumulative message on purpose.
3. **Cross-check** — buffer against `text_end` and `message_end`.
4. **Wait for `agent_settled`** — `agent_end` is not the end.
5. **Read the failure in the stream** — exit 0 is not success in JSON mode.

## Sources, execution mode and limitations

**Execution mode:** **Run** — the shipped binary in `--mode json` with a scripted model.

**Evidence scope:** `shipped_binary`. The event vocabulary and framing are the real ones.

### What was read or run

- **Ran** `drivers/ch33-json.ts`: six shipped-binary JSON runs (header, U+2028, lifecycle, long answer, reconstruction, tool, failure).
- **Ran** `ch33-json/json-stream.test.ts` (11 tests).
- **Read** `examples/evidence.json`, chapter 33 rows (11 claims).

### Limitations

- **Representative records, not every compaction/retry variant** (`ch33-lim1`).
- **`extension_error` is RPC-only** — not a JSON-mode record.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
